In [ ]:
from datetime import datetime
from getpass import getpass

rdm_url = 'http://localhost:5001/'
admin_rdm_url = 'http://localhost:8001/'
idp_name_1 = 'FakeCAS'
idp_username_1 = None
idp_password_1 = None
institution_name = 'Massachusetts Institute of Technology [Test]'
project_name = None
project_url = None
engine_name = None
engine_name_updated = None
gateway_base_url = 'http://192.168.168.167:8088/'
default_result_path = None
close_on_fail = False
transition_timeout = 30000
browser_type = 'chromium'
ignore_https_errors = False
is_integrated_admin = True

In [ ]:
import tempfile

if idp_username_1 is None:
    idp_username_1 = input(prompt=f'Username for {idp_name_1}')
if idp_password_1 is None:
    idp_password_1 = getpass(prompt=f'Password for {idp_username_1}@{idp_name_1}')

if project_name is None:
    project_name = datetime.now().strftime('TEST-WORKFLOW-%Y%m%d%H%M')
if engine_name is None:
    engine_name = datetime.now().strftime('TEST-ENGINE-%Y%m%d%H%M')
if engine_name_updated is None:
    engine_name_updated = engine_name + '-Updated'

# RDMワークフローエンジンの登録と編集

- サブシステム名: 管理機能
- ページ/アドオン: RDMワークフロー
- 機能分類: ワークフローエンジン管理
- シナリオ名: エンジンの登録・編集
- 用意するテストデータ: 管理者アカウント、機関名、ゲートウェイURL
- 事前条件: 管理者権限を持つアカウントが存在すること

In [ ]:
work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

In [ ]:
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path, ignore_https_errors=ignore_https_errors)

## 新規タブを開き、GRDMトップページを表示する

GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url)

    # 同意する をクリック
    await page.locator('//button[text() = "同意する"]').click()

    # 同意する が表示されなくなったことを確認
    await expect(page.locator('//button[text() = "同意する"]')).to_have_count(0, timeout=500)

await run_pw(_step, new_page=True)

## ログイン情報を用いてGakuNin RDMにログインする

GRDMダッシュボードが表示されること

In [ ]:
async def _step(page):
    await grdm.login(page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout)
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)

## ダッシュボードから「新規プロジェクト作成」をクリックする

指定したプロジェクトが存在しない場合、新規プロジェクトが作成されること

In [ ]:
async def _step(page):
    await grdm.ensure_project_exists(page, project_name, transition_timeout=transition_timeout)

await run_pw(_step)

## ダッシュボードのプロジェクト一覧から作成したプロジェクトをクリックする

プロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):
    global project_url
    await page.locator(f'//*[@data-test-dashboard-item-title and text() = "{project_name}"]').click()
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)
    project_url = page.url

await run_pw(_step)

In [ ]:
from urllib.parse import urlparse

project_id = urlparse(project_url).path.strip('/')
project_id

## GakuNin RDM管理者ページのURLを開く

ログイン画面が表示されること

In [ ]:
async def _step(page):
    await page.goto(admin_rdm_url)
    await expect(page.locator('.login-logo')).to_be_visible(timeout=transition_timeout)

await run_pw(_step, new_page=True)

## ログイン情報を用いてGakuNin RDMにログインする

(IdPに関するログイン情報が与えられた場合、)
GakuNin Embeded DSのプルダウンを展開し、IdPリストから指定されたIdPを選択する。その後、アカウントのID/Passwordを入力して「Login」ボタンを押下する。

(IdPが指定されていない場合、)
CASのログイン操作を実施する。

In [ ]:
import scripts.grdm
importlib.reload(scripts.grdm)

async def _step(page):
    await scripts.grdm.login_as_admin(
        page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout
    )
    await expect(page.locator('//*[contains(@class, "btn-danger") and contains(text(), "ログアウト")]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## サイドメニューから「RDMワークフロー」をクリックする

(統合管理者の場合)機関一覧が表示されること 
(機関管理者の場合)RDMワークフローエンジン管理ページが表示されること

In [ ]:
async def _step(page):
    await page.locator('//span[text()="RDMワークフロー"]').click()
    if is_integrated_admin:
        await expect(page.locator('//table//th[contains(text(), "機関")]')).to_be_visible(timeout=transition_timeout)
    else:
        await expect(page.locator('#id_label')).to_be_visible(timeout=transition_timeout)
        await expect(page.locator('#id_gateway_base_url')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## (統合管理者の場合)機関一覧から対象の機関名をクリックする

ワークフローエンジン管理ページが表示されること

In [ ]:
import traceback

async def _step(page):
    if is_integrated_admin:
        while True:
            link = page.locator(f'//table//a[text() = "{institution_name}"]')
            try:
                await expect(link).to_be_visible()
            except:
                traceback.print_exc()
                print('Search next page...')
                await page.locator('.pagination').locator('//a[i[contains(@class, "fa-angle-right")]]').first.click()
                await expect(page.locator('//table//th[contains(text(), "機関")]')).to_be_visible(timeout=transition_timeout)
                continue
            await link.click()
            break
    await expect(page.locator('#id_label')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('#id_gateway_base_url')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「名前」と「ゲートウェイのベースURL」を入力し、「ワークフローエンジンを登録」をクリックする

エンジンが登録され、エンジン一覧に表示されること

In [ ]:
async def _step(page):
    await page.locator('#id_label').fill(engine_name)
    await page.locator('#id_gateway_base_url').fill(gateway_base_url)
    await page.locator('//button[contains(text(), "ワークフローエンジンを登録")]').click()
    engine_item = page.locator(f'//table//td//strong[contains(text(), "{engine_name}")]')
    await expect(engine_item).to_be_visible(timeout=transition_timeout)
    await engine_item.scroll_into_view_if_needed()

await run_pw(_step)

## 登録したエンジンの「編集」リンクをクリックする

エンジン編集ページが表示され、登録した名前がフォームに表示されていること

In [ ]:
async def _step(page):
    row = page.locator(f'//table//tr[.//td//strong[contains(text(), "{engine_name}")]]')
    await row.locator('//a[contains(text(), "編集")]').click()
    await expect(page.locator('#id_label')).to_have_value(engine_name, timeout=transition_timeout)

await run_pw(_step)

## 「名前」を変更し、「アップロード許可ノードID」にプロジェクトIDを入力し、「変更を保存」をクリックする

変更が保存され、エンジン一覧に更新後の名前が表示されること

In [ ]:
async def _step(page):
    await page.locator('#id_label').fill(engine_name_updated)
    await page.locator('#id_upload_whitelist_node_ids_input').fill(project_id)
    await page.locator('//button[contains(text(), "変更を保存")]').click()
    await expect(page.locator(f'//table//td//strong[contains(text(), "{engine_name_updated}")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 後始末

In [ ]:
await finish_pw_context(screenshot=False, last_path=default_result_path)

In [ ]:
!rm -fr {work_dir}